In [ ]:
import duckdb
import pandas as pd

pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

con = duckdb.connect(r'data\gold.duckdb', read_only=True)
print('Connected to gold.duckdb')

def q(sql):
    return con.sql(sql).df()

## Data Coverage

In [ ]:
# Matches by type
q("""
SELECT match_type, count(*) AS matches
FROM main_marts.fct_match_results
GROUP BY match_type
ORDER BY matches DESC
""")

In [ ]:
# Matches per year (last 10 years)
q("""
SELECT year, match_type, count(*) AS matches
FROM main_marts.fct_match_results
WHERE year >= year(today()) - 10
GROUP BY year, match_type
ORDER BY year DESC, matches DESC
""")

In [ ]:
# Most recent matches loaded
q("""
SELECT match_id, match_type, match_date, team_1, team_2, outcome_winner
FROM main_marts.fct_match_results
ORDER BY match_date DESC
LIMIT 20
""")

## Batting

In [ ]:
# Top 20 ODI run scorers (min 50 innings)
q("""
SELECT player_name, matches, innings, total_runs, highest_score,
       batting_avg, batting_sr, centuries, fifties
FROM main_marts.mart_batting_career
WHERE match_type = 'odi'
  AND innings >= 50
ORDER BY total_runs DESC
LIMIT 20
""")

In [ ]:
# Top T20 batters by strike rate (min 30 innings)
q("""
SELECT player_name, innings, total_runs, batting_avg, batting_sr,
       total_sixes, total_fours
FROM main_marts.mart_batting_career
WHERE match_type = 't20'
  AND innings >= 30
ORDER BY batting_sr DESC
LIMIT 20
""")

In [ ]:
# Career stats for a specific player — change name as needed
q("""
SELECT player_name, match_type, matches, innings, total_runs,
       batting_avg, batting_sr, centuries, fifties
FROM main_marts.mart_batting_career
WHERE player_name = 'V Kohli'
ORDER BY match_type
""")

In [ ]:
# Batting average trend by year for a player (ODIs)
q("""
SELECT year,
       count(distinct match_id) AS matches,
       sum(runs) AS runs,
       round(sum(runs)::double / nullif(sum(dismissals), 0), 2) AS avg
FROM main_intermediate.int_batting_by_innings
WHERE player_name = 'V Kohli'
  AND match_type = 'odi'
  AND NOT super_over
GROUP BY year
ORDER BY year
""")

## Bowling

In [ ]:
# Top 20 ODI wicket takers (min 50 innings bowled)
q("""
SELECT player_name, matches, innings_bowled, total_wickets,
       bowling_avg, economy, bowling_sr, five_wicket_hauls
FROM main_marts.mart_bowling_career
WHERE match_type = 'odi'
  AND innings_bowled >= 50
ORDER BY total_wickets DESC
LIMIT 20
""")

In [ ]:
# Best T20 economy rates (min 50 innings)
q("""
SELECT player_name, innings_bowled, total_wickets, economy, bowling_avg, bowling_sr
FROM main_marts.mart_bowling_career
WHERE match_type = 't20'
  AND innings_bowled >= 50
ORDER BY economy
LIMIT 20
""")

In [ ]:
# Economy by over number in T20s (powerplay vs middle vs death)
q("""
SELECT over_number,
       count(*) AS balls,
       sum(runs_total) AS runs,
       round(sum(runs_total) * 6.0 / nullif(
           count(*) - sum(extras_wides::integer), 0), 2) AS economy
FROM main_marts.fct_deliveries
WHERE match_type = 't20'
  AND innings_number IN (1, 2)
GROUP BY over_number
ORDER BY over_number
""")

## Match Results

In [ ]:
# India ODI win/loss record
q("""
SELECT team_1 AS team, count(*) AS played,
       sum((outcome_winner = team_1)::integer) AS won,
       sum((outcome_winner = team_2)::integer) AS lost,
       sum(is_tie::integer) AS tied,
       sum((NOT has_result)::integer) AS no_result,
       round(sum((outcome_winner = team_1)::integer) * 100.0
           / nullif(sum(has_result::integer), 0), 1) AS win_pct
FROM main_marts.fct_match_results
WHERE match_type = 'odi'
  AND team_1 = 'India'
UNION ALL
SELECT team_2, count(*),
       sum((outcome_winner = team_2)::integer),
       sum((outcome_winner = team_1)::integer),
       sum(is_tie::integer),
       sum((NOT has_result)::integer),
       round(sum((outcome_winner = team_2)::integer) * 100.0
           / nullif(sum(has_result::integer), 0), 1)
FROM main_marts.fct_match_results
WHERE match_type = 'odi'
  AND team_2 = 'India'
""")

In [ ]:
# India vs Australia head-to-head (all formats)
q("""
SELECT match_type, count(*) AS played,
       sum((outcome_winner = 'India')::integer) AS india_wins,
       sum((outcome_winner = 'Australia')::integer) AS aus_wins,
       sum(is_tie::integer) AS ties
FROM main_marts.fct_match_results
WHERE has_result
  AND (
    (team_1 = 'India' AND team_2 = 'Australia') OR
    (team_1 = 'Australia' AND team_2 = 'India')
  )
GROUP BY match_type
ORDER BY match_type
""")

In [ ]:
# Toss advantage: does winning the toss help?
q("""
SELECT match_type, toss_decision, count(*) AS matches,
       sum((toss_winner = outcome_winner)::integer) AS toss_winner_won,
       round(sum((toss_winner = outcome_winner)::integer) * 100.0
           / nullif(count(*), 0), 1) AS toss_win_pct
FROM main_marts.fct_match_results
WHERE has_result
  AND match_type IN ('t20', 'odi', 'test')
GROUP BY match_type, toss_decision
ORDER BY match_type, toss_decision
""")

In [ ]:
# Most player of the match awards
q("""
SELECT player_of_match AS player, match_type, count(*) AS awards
FROM main_marts.fct_match_results
WHERE player_of_match IS NOT NULL
GROUP BY player_of_match, match_type
ORDER BY awards DESC
LIMIT 20
""")

## IPL Analysis

In [ ]:
# V Kohli IPL career batting stats (computed from ball-by-ball data)
q("""
WITH ipl_innings AS (
    SELECT
        d.match_id,
        d.innings_number,
        sum(d.runs_batter)                                                              AS runs,
        sum(CASE WHEN NOT d.extras_wides THEN 1 ELSE 0 END)                            AS balls_faced,
        sum(CASE WHEN d.runs_batter = 4 AND NOT d.runs_non_boundary THEN 1 ELSE 0 END) AS fours,
        sum(CASE WHEN d.runs_batter = 6 AND NOT d.runs_non_boundary THEN 1 ELSE 0 END) AS sixes,
        max(CASE WHEN d.is_wicket AND d.wicket_player_out = 'V Kohli' THEN 1 ELSE 0 END) AS dismissed
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.batter = 'V Kohli'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
    GROUP BY d.match_id, d.innings_number
)
SELECT
    'V Kohli'                                                   AS player_name,
    count(distinct match_id)                                    AS matches,
    count(*)                                                    AS innings,
    sum(runs)                                                   AS total_runs,
    max(runs)                                                   AS highest_score,
    round(sum(runs)::double / nullif(sum(dismissed), 0), 2)    AS batting_avg,
    round(sum(runs) * 100.0 / nullif(sum(balls_faced), 0), 2)  AS strike_rate,
    sum(CASE WHEN runs >= 100 THEN 1 ELSE 0 END)               AS centuries,
    sum(CASE WHEN runs >= 50 AND runs < 100 THEN 1 ELSE 0 END) AS fifties,
    sum(fours)                                                  AS total_fours,
    sum(sixes)                                                  AS total_sixes
FROM ipl_innings
""")

In [ ]:
# V Kohli IPL stats by season
q("""
WITH ipl_innings AS (
    SELECT
        d.match_id,
        d.innings_number,
        d.batting_team,
        m.season,
        sum(d.runs_batter)                                                              AS runs,
        sum(CASE WHEN NOT d.extras_wides THEN 1 ELSE 0 END)                            AS balls_faced,
        max(CASE WHEN d.is_wicket AND d.wicket_player_out = 'V Kohli' THEN 1 ELSE 0 END) AS dismissed
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.batter = 'V Kohli'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
    GROUP BY d.match_id, d.innings_number, d.batting_team, m.season
)
SELECT
    season,
    batting_team,
    count(distinct match_id)                                   AS matches,
    count(*)                                                   AS innings,
    sum(runs)                                                  AS runs,
    max(runs)                                                  AS highest,
    round(sum(runs)::double / nullif(sum(dismissed), 0), 2)   AS avg,
    round(sum(runs) * 100.0 / nullif(sum(balls_faced), 0), 2) AS sr,
    sum(CASE WHEN runs >= 50 THEN 1 ELSE 0 END)               AS fifties_plus
FROM ipl_innings
GROUP BY season, batting_team
ORDER BY season
""")

In [ ]:
# V Kohli IPL innings log (most recent first)
q("""
SELECT
    m.season,
    d.match_date,
    d.batting_team,
    CASE WHEN m.team_1 = d.batting_team THEN m.team_2 ELSE m.team_1 END AS vs,
    sum(d.runs_batter)                                                              AS runs,
    sum(CASE WHEN NOT d.extras_wides THEN 1 ELSE 0 END)                            AS balls_faced,
    round(sum(d.runs_batter) * 100.0
        / nullif(sum(CASE WHEN NOT d.extras_wides THEN 1 ELSE 0 END), 0), 1)       AS strike_rate,
    sum(CASE WHEN d.runs_batter = 4 AND NOT d.runs_non_boundary THEN 1 ELSE 0 END) AS fours,
    sum(CASE WHEN d.runs_batter = 6 AND NOT d.runs_non_boundary THEN 1 ELSE 0 END) AS sixes,
    NOT max(CASE WHEN d.is_wicket AND d.wicket_player_out = 'V Kohli' THEN 1 ELSE 0 END)::boolean AS not_out
FROM main_marts.fct_deliveries d
JOIN main_marts.fct_match_results m USING (match_id)
WHERE d.batter = 'V Kohli'
  AND m.event_name ILIKE '%indian premier league%'
  AND d.innings_number IN (1, 2)
GROUP BY m.season, d.match_id, d.innings_number, d.match_date, d.batting_team, m.team_1, m.team_2
ORDER BY d.match_date DESC
""")